# 📈 Previsão de Demanda e Planejamento de Estoque — SKU MI-006

Projeto de previsão de demanda diária com foco em planejamento de estoque.

**Dataset:** FMCG Daily Sales (2022–2024) — laticínios (Polônia)

**Série modelada:** SKU YO-029, vendas diárias agregadas

**Modelos:** Prophet (baseline) e XGBoost

## 1. Bibliotecas:

In [7]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Yukio Kitamura\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
%pip install prophet

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Yukio Kitamura\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Yukio Kitamura\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
%pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Yukio Kitamura\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

## 2. Carregamento dos Dados

Leitura do arquivo CSV original.

In [12]:
df = pd.read_csv("FMCG_2022_2024.csv")

### 2.1. ANÁLISE EXPLORATÓRIA DOS DADOS:

In [13]:
# DIMENSÕES DO DATASET: (LINHAS, COLUNAS)
print(df.shape)

(190757, 14)


In [14]:
# PRIMEIRAS LINHAS DO DATASET
df.head()

,date,sku,brand,segment,category,channel,region,pack_type,price_unit,promotion_flag,delivery_days,stock_available,delivered_qty,units_sold
0,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Retail,PL-Central,Multipack,2.38,0,1,141,128,9
1,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Retail,PL-North,Single,1.55,1,3,0,129,0
2,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Retail,PL-South,Carton,4.00,0,5,118,161,8
3,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Discount,PL-Central,Single,5.16,0,2,81,114,7
4,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Discount,PL-North,Single,7.66,0,4,148,204,12


In [15]:
# NOMES DAS COLUNAS E TIPOS DE DADO
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 190757 entries, 0 to 190756
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   date             190757 non-null  str    
 1   sku              190757 non-null  str    
 2   brand            190757 non-null  str    
 3   segment          190757 non-null  str    
 4   category         190757 non-null  str    
 5   channel          190757 non-null  str    
 6   region           190757 non-null  str    
 7   pack_type        190757 non-null  str    
 8   price_unit       190757 non-null  float64
 9   promotion_flag   190757 non-null  int64  
 10  delivery_days    190757 non-null  int64  
 11  stock_available  190757 non-null  int64  
 12  delivered_qty    190757 non-null  int64  
 13  units_sold       190757 non-null  int64  
dtypes: float64(1), int64(5), str(8)
memory usage: 20.4 MB


### 2.2 Conversão da coluna de data

A coluna `date` foi carregada como texto. Vamos convertê-la para o tipo
data do pandas.

In [16]:
df["date"] = pd.to_datetime(df["date"])

In [17]:
df["date"].dtype

dtype('<M8[us]')

### 2.3 Período coberto pelos dados

Vamos identificar a primeira e a última data presentes no dataset.

In [18]:
df["date"].min()

Timestamp('2022-01-21 00:00:00')

In [19]:
df["date"].max()

Timestamp('2024-12-31 00:00:00')

### 2.4 Quantidade de registros por SKU

Vamos contar quantas linhas existem para cada SKU no dataset. Essa contagem
mostra a distribuição dos registros.

In [20]:
df["sku"].value_counts()

sku
MI-006    8221
MI-026    8216
YO-029    7909
YO-005    7893
YO-012    7719
RE-004    7643
YO-014    7630
YO-001    7574
RE-007    7489
RE-015    7366
MI-023    7202
JU-021    6943
YO-009    6778
SN-027    6435
YO-003    6341
MI-022    6249
YO-016    5991
RE-025    5990
SN-010    5870
RE-017    5748
SN-013    5501
YO-020    5416
SN-019    5229
MI-002    5067
MI-011    4863
YO-024    4850
MI-008    4777
SN-028    4749
YO-018    4606
SN-030    4492
Name: count, dtype: int64

### 2.5 Volume vendido por SKU

A contagem anterior mostrou quantos registros existem para cada SKU.
Agora vamos somar `units_sold` por SKU para comparar o volume registrado.

In [21]:
# AGRUPANDO OS DADOS POR SKU E SOMANDO AS VENDAS DE UNIDADES, ORDENANDO EM ORDEM DECRESCENTE
df.groupby("sku")["units_sold"].sum().sort_values(ascending=False)

sku
YO-029    177798
YO-005    176095
YO-012    170575
MI-026    152324
RE-004    149559
YO-014    148845
RE-007    146705
YO-001    146646
YO-009    145236
RE-015    144534
YO-003    137173
MI-023    133624
YO-016    132999
MI-006    132623
JU-021    124349
YO-020    121833
RE-025    121684
SN-010    121499
RE-017    116352
SN-027    115386
SN-019    109593
YO-024    109519
SN-013    103410
SN-028    101706
MI-022    101115
YO-018     99863
SN-030     95731
MI-002     95534
MI-011     84626
MI-008     82888
Name: units_sold, dtype: int64

## 3. Seleção da SKU para modelagem

A SKU YO-029 foi escolhida por apresentar o maior total de `units_sold`
no dataset.

In [22]:
# FILTRAR OS REGISTROS DA SKU YO-029
df_yo = df[df["sku"] == "YO-029"].copy()

### 3.1 Conferência do filtro

Vamos verificar a quantidade de registros da YO-029 e confirmar que não
restaram outras SKUs no recorte.

In [23]:
df_yo.shape

(7909, 14)

In [24]:
df_yo["sku"].unique()

<StringArray>
['YO-029']
Length: 1, dtype: str

### 3.2 Verificação de linhas por data

Vamos conferir quantas linhas existem para cada data da YO-029. Se houver
mais de uma linha por dia, precisaremos agregar por data antes de modelar.

In [25]:
# LINHAS POR DATA: A CONTAGEM VARIADA INDICA VÁRIOS REGISTROS NO MESMO DIA
df_yo.groupby("date").size().value_counts().sort_index()

4      5
5     36
6    119
7    277
8    353
9    248
Name: count, dtype: int64

#### 3.2.1 Agregação diária da YO-029

Cada data tem vários registros, relacionados a diferentes dimensões do
dataset. Vamos somar `units_sold` por data para obter a venda diária total
e usar o máximo de `promotion_flag` para marcar se houve promoção naquele dia.

In [26]:
# AGREGAR VENDAS E PROMOÇÕES POR DATA
df_yo_diario = (
    df_yo.groupby("date")
    .agg({
        "units_sold": "sum",
        "promotion_flag": "max"
    })
    .sort_index()
)

In [27]:
df_yo_diario.shape

(1038, 2)

In [28]:
df_yo_diario.columns

Index(['units_sold', 'promotion_flag'], dtype='str')

In [29]:
df_yo_diario.index.is_unique

True

In [30]:
df_yo_diario.index.min()

Timestamp('2022-02-28 00:00:00')

In [31]:
df_yo_diario.index.max()

Timestamp('2024-12-31 00:00:00')

### Continuidade da série diária

Vamos comparar o número de datas observadas com o total de dias corridos
entre a primeira e a última data. Isso identifica lacunas na série.

In [32]:
# COMPARAR DATAS OBSERVADAS COM O TOTAL DE DIAS DO INTERVALO
dias_observados = len(df_yo_diario)
dias_no_intervalo = (
    df_yo_diario.index.max() - df_yo_diario.index.min()
).days + 1

dias_observados, dias_no_intervalo

(1038, 1038)

## 4. Separação temporal

Vamos reservar o período de outubro a dezembro de 2024 para teste.
O modelo será treinado apenas com dados anteriores a 1º de outubro de 2024.

In [33]:
data_separacao = "2024-10-01"

### 4.1 Separação dos períodos de treino e teste

O treino será usado para ajustar os modelos. O teste, de outubro a dezembro
de 2024, será reservado para avaliar as previsões.

In [34]:
df_treino = df_yo_diario.loc[
    df_yo_diario.index < data_separacao
].copy()

In [35]:
df_teste = df_yo_diario.loc[
    df_yo_diario.index >= data_separacao
].copy()

In [36]:
print("TREINO:", len(df_treino), df_treino.index.min(), df_treino.index.max())
print("TESTE:", len(df_teste), df_teste.index.min(), df_teste.index.max())

TREINO: 946 2022-02-28 00:00:00 2024-09-30 00:00:00
TESTE: 92 2024-10-01 00:00:00 2024-12-31 00:00:00


### 4.2 Datas de origem das previsões

Faremos uma previsão de 30 dias para cada mês do período de teste.
As datas de origem correspondem ao início de outubro, novembro e dezembro
de 2024. Em cada origem, o modelo usará apenas dados anteriores à previsão.

In [37]:
datas_origem = [
    "2024-10-01",
    "2024-11-01",
    "2024-12-01"
]

In [38]:
datas_origem

['2024-10-01', '2024-11-01', '2024-12-01']

### 4.3 Períodos mensais de avaliação

Vamos conferir os intervalos de teste previstos para cada origem:
outubro, novembro e dezembro de 2024. Cada previsão cobrirá o mês-calendário
correspondente.

In [39]:
for data_origem in datas_origem:
    inicio = pd.Timestamp(data_origem)
    fim = inicio + pd.offsets.MonthEnd(0)
    print(data_origem, inicio.date(), "a", fim.date())

2024-10-01 2024-10-01 a 2024-10-31
2024-11-01 2024-11-01 a 2024-11-30
2024-12-01 2024-12-01 a 2024-12-31


### 4.4 Primeiro período de avaliação: outubro de 2024

Para a previsão de outubro, o treino usa os dados anteriores a 1º de outubro.
O teste contém as vendas observadas de 1º a 31 de outubro, usadas depois
para avaliar as previsões.

In [40]:
data_origem = "2024-10-01"

treino_outubro = df_yo_diario.loc[
    df_yo_diario.index < data_origem
].copy()

In [41]:
teste_outubro = df_yo_diario.loc[
    df_yo_diario.index >= data_origem
].loc[:"2024-10-31"].copy()

##### Conferência dos recortes de outubro

O treino deve terminar em 30/09/2024 e o teste deve cobrir 01/10 a
31/10/2024.

In [42]:
print("TREINO:", len(treino_outubro), treino_outubro.index.min(), treino_outubro.index.max())
print("TESTE:", len(teste_outubro), teste_outubro.index.min(), teste_outubro.index.max())

TREINO: 946 2022-02-28 00:00:00 2024-09-30 00:00:00
TESTE: 31 2024-10-01 00:00:00 2024-10-31 00:00:00


### 4.5 Período de avaliação: novembro de 2024

Para a previsão de novembro, o treino inclui os dados observados até 31 de
outubro. O teste contém as vendas de 1º a 30 de novembro.

In [43]:
data_origem = "2024-11-01"

treino_novembro = df_yo_diario.loc[
    df_yo_diario.index < data_origem
].copy()

In [44]:
teste_novembro = df_yo_diario.loc[
    data_origem:"2024-11-30"
].copy()

##### Conferência dos recortes de novembro

O treino deve terminar em 31/10/2024 e o teste deve cobrir 01/11 a
30/11/2024.

In [45]:
print("TREINO:", len(treino_novembro), treino_novembro.index.min(), treino_novembro.index.max())
print("TESTE:", len(teste_novembro), teste_novembro.index.min(), teste_novembro.index.max())

TREINO: 977 2022-02-28 00:00:00 2024-10-31 00:00:00
TESTE: 30 2024-11-01 00:00:00 2024-11-30 00:00:00


### 4.6 Período de avaliação: dezembro de 2024

Para a previsão de dezembro, o treino inclui os dados observados até 30 de
novembro. O teste contém as vendas de 1º a 31 de dezembro.

In [46]:
data_origem = "2024-12-01"

treino_dezembro = df_yo_diario.loc[
    df_yo_diario.index < data_origem
].copy()

In [47]:
teste_dezembro = df_yo_diario.loc[
    data_origem:"2024-12-31"
].copy()

###### Conferência dos recortes de dezembro

O treino deve terminar em 30/11/2024 e o teste deve cobrir 01/12 a
31/12/2024.

In [48]:
print("TREINO:", len(treino_dezembro), treino_dezembro.index.min(), treino_dezembro.index.max())
print("TESTE:", len(teste_dezembro), teste_dezembro.index.min(), teste_dezembro.index.max())

TREINO: 1007 2022-02-28 00:00:00 2024-11-30 00:00:00
TESTE: 31 2024-12-01 00:00:00 2024-12-31 00:00:00


## 5. Prophet: preparação dos dados de outubro

O Prophet espera uma coluna `ds` com as datas e uma coluna `y` com os valores
a prever. Vamos preparar o treino de outubro nesse formato. Não incluiremos
promoções como variável, pois os valores futuros não são conhecidos no
cenário definido.

### 5.1. Outubro:

In [49]:
dados_prophet_treino_outubro = (
    treino_outubro[["units_sold"]]
    .reset_index()
    .rename(columns={"date": "ds", "units_sold": "y"})
)

##### Dados reais de outubro para avaliação

Vamos organizar as datas e vendas observadas do período de teste no formato
`ds` e `y`. Esses valores serão usados somente para avaliar as previsões.

In [50]:
dados_prophet_teste_outubro = (
    teste_outubro[["units_sold"]]
    .reset_index()
    .rename(columns={"date": "ds", "units_sold": "y"})
)

##### Conferência do formato de treino

Vamos verificar as colunas e as primeiras linhas dos dados preparados para o
Prophet. A coluna `ds` representa a data e `y`, as vendas.

In [51]:
dados_prophet_treino_outubro.head()

,ds,y
0,2022-02-28,141
1,2022-03-01,238
2,2022-03-02,196
3,2022-03-03,138
4,2022-03-04,112


##### Conferência dos dados de teste

Vamos verificar se o período reservado para avaliação também está no formato
`ds` e `y`, cobrindo outubro de 2024.

In [52]:
dados_prophet_teste_outubro.head()

,ds,y
0,2024-10-01,133
1,2024-10-02,163
2,2024-10-03,153
3,2024-10-04,140
4,2024-10-05,135


##### Ajuste do Prophet para outubro

Vamos ajustar o modelo usando somente os dados disponíveis até
30/09/2024. As vendas de outubro ficarão reservadas para avaliar as
previsões.

##### Ajuste do modelo

O Prophet será ajustado com os dados de treino de outubro, nas colunas `ds`
(data) e `y` (vendas). Não usaremos `promotion_flag`, pois o calendário de
promoções futuras não é conhecido no cenário definido.

In [53]:
modelo_prophet_outubro = Prophet()
modelo_prophet_outubro.fit(dados_prophet_treino_outubro)

C:\Users\Yukio Kitamura\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
21:16:22 - cmdstanpy - INFO - Chain [1] start processing
21:16:22 - cmdstanpy - INFO - Chain [1] done processing


##### Datas a prever em outubro

Vamos criar uma tabela com as datas de 1º a 31 de outubro de 2024 para
solicitar uma previsão para cada dia do mês.

In [54]:
datas_futuras_outubro = pd.DataFrame({
    "ds": pd.date_range("2024-10-01", "2024-10-31", freq="D")
})

##### Previsão de vendas para outubro

Vamos aplicar o Prophet às datas de outubro e guardar as previsões junto
com os limites inferior e superior estimados pelo modelo.

In [55]:
previsao_prophet_outubro = modelo_prophet_outubro.predict(
    datas_futuras_outubro
)

##### Comparação entre vendas reais e previstas

Vamos juntar as vendas observadas em outubro com a previsão do Prophet,
usando a data como chave. Essa tabela será a base para avaliar o modelo.

In [56]:
comparacao_prophet_outubro = dados_prophet_teste_outubro.merge(
    previsao_prophet_outubro[["ds", "yhat"]],
    on="ds",
    how="left"
)

##### Conferência da comparação de outubro

Vamos verificar as primeiras linhas e as dimensões da tabela que reúne
as vendas reais e as previsões do Prophet.

In [57]:
comparacao_prophet_outubro.head()

,ds,y,yhat
0,2024-10-01,133,118.369090
1,2024-10-02,163,120.981838
2,2024-10-03,153,117.923662
3,2024-10-04,140,119.703243
4,2024-10-05,135,114.721867


In [58]:
comparacao_prophet_outubro.shape

(31, 3)

##### Métricas do Prophet em outubro

O MAE mede o erro absoluto médio. O RMSE também resume os erros, mas
penaliza mais os erros grandes. As duas métricas são expressas em unidades
vendidas por dia.

In [59]:
mae_prophet_outubro = mean_absolute_error(
    comparacao_prophet_outubro["y"],
    comparacao_prophet_outubro["yhat"]
)

In [60]:
rmse_prophet_outubro = mean_squared_error(
    comparacao_prophet_outubro["y"],
    comparacao_prophet_outubro["yhat"]
) ** 0.5

In [61]:
print("MAE:", mae_prophet_outubro)
print("RMSE:", rmse_prophet_outubro)

MAE: 23.60360762944198
RMSE: 29.99377336288883


### 5.2. Novembro:

##### Prophet: preparação do treino de novembro

O modelo de novembro será ajustado com os dados disponíveis até 31/10/2024.
Vamos organizar a série nas colunas `ds` (data) e `y` (vendas).

In [62]:
dados_prophet_treino_novembro = (
    treino_novembro[["units_sold"]]
    .reset_index()
    .rename(columns={"date": "ds", "units_sold": "y"})
)

##### Dados reais de novembro para avaliação

Vamos organizar as datas e vendas observadas de novembro nas colunas `ds`
(data) e `y` (vendas).

In [63]:
dados_prophet_teste_novembro = (
    teste_novembro[["units_sold"]]
    .reset_index()
    .rename(columns={"date": "ds", "units_sold": "y"})
)

##### Ajuste do Prophet para novembro

O modelo será treinado com os dados disponíveis até 31/10/2024.
As vendas de novembro ficarão reservadas para avaliar as previsões.

In [64]:
modelo_prophet_novembro = Prophet()
modelo_prophet_novembro.fit(dados_prophet_treino_novembro)

21:16:24 - cmdstanpy - INFO - Chain [1] start processing
21:16:24 - cmdstanpy - INFO - Chain [1] done processing


##### Datas a prever em novembro

Vamos criar as datas de 1º a 30 de novembro de 2024 para solicitar uma
previsão diária do Prophet.

In [65]:
datas_futuras_novembro = pd.DataFrame({
    "ds": pd.date_range("2024-11-01", "2024-11-30", freq="D")
})

##### Previsão do Prophet para novembro

Vamos prever as vendas diárias de 1º a 30 de novembro de 2024.

In [66]:
previsao_prophet_novembro = modelo_prophet_novembro.predict(
    datas_futuras_novembro
)

##### Comparação entre vendas reais e previstas em novembro

Vamos combinar os valores reais do teste com a previsão do Prophet para
preparar a avaliação do modelo.

In [67]:
comparacao_prophet_novembro = dados_prophet_teste_novembro.merge(
    previsao_prophet_novembro[["ds", "yhat"]],
    on="ds",
    how="left"
)

In [68]:
comparacao_prophet_novembro.head()

,ds,y,yhat
0,2024-11-01,122,119.305952
1,2024-11-02,139,115.350357
2,2024-11-03,137,124.511144
3,2024-11-04,126,117.147390
4,2024-11-05,129,118.287517


##### Métricas do Prophet em novembro

Vamos calcular o MAE e o RMSE entre as vendas reais e as previsões para
novembro. As métricas são expressas em unidades vendidas por dia.

In [69]:
mae_prophet_novembro = mean_absolute_error(
    comparacao_prophet_novembro["y"],
    comparacao_prophet_novembro["yhat"]
)

In [70]:
rmse_prophet_novembro = mean_squared_error(
    comparacao_prophet_novembro["y"],
    comparacao_prophet_novembro["yhat"]
) ** 0.5

In [71]:
print("MAE:", mae_prophet_novembro)
print("RMSE:", rmse_prophet_novembro)

MAE: 25.893052066965165
RMSE: 32.44500542681185


### 5.3. Dezembro:

##### Prophet: preparação do treino de dezembro

O modelo de dezembro será ajustado com os dados disponíveis até 30/11/2024.
Vamos organizar a série nas colunas `ds` (data) e `y` (vendas).

In [72]:
dados_prophet_treino_dezembro = (
    treino_dezembro[["units_sold"]]
    .reset_index()
    .rename(columns={"date": "ds", "units_sold": "y"})
)

##### Dados reais de dezembro para avaliação

Vamos organizar as datas e vendas observadas de dezembro nas colunas `ds`
(data) e `y` (vendas). Esses valores serão usados somente na avaliação.

In [73]:
dados_prophet_teste_dezembro = (
    teste_dezembro[["units_sold"]]
    .reset_index()
    .rename(columns={"date": "ds", "units_sold": "y"})
)

##### Ajuste do Prophet para dezembro

O modelo será treinado com os dados disponíveis até 30/11/2024.
As vendas de dezembro ficarão reservadas para avaliar as previsões.

In [74]:
modelo_prophet_dezembro = Prophet()
modelo_prophet_dezembro.fit(dados_prophet_treino_dezembro)

21:16:25 - cmdstanpy - INFO - Chain [1] start processing
21:16:25 - cmdstanpy - INFO - Chain [1] done processing


##### Datas a prever em dezembro

Vamos criar as datas de 1º a 31 de dezembro de 2024 para solicitar uma
previsão diária do Prophet.

In [75]:
datas_futuras_dezembro = pd.DataFrame({
    "ds": pd.date_range("2024-12-01", "2024-12-31", freq="D")
})

##### Previsão do Prophet para dezembro

Vamos prever as vendas diárias de 1º a 31 de dezembro de 2024.

In [76]:
previsao_prophet_dezembro = modelo_prophet_dezembro.predict(
    datas_futuras_dezembro
)

##### Comparação entre vendas reais e previstas em dezembro

Vamos combinar os valores reais do teste com a previsão do Prophet para
preparar a avaliação do modelo.

In [77]:
comparacao_prophet_dezembro = dados_prophet_teste_dezembro.merge(
    previsao_prophet_dezembro[["ds", "yhat"]],
    on="ds",
    how="left"
)

In [78]:
comparacao_prophet_dezembro.head()

,ds,y,yhat
0,2024-12-01,191,122.778210
1,2024-12-02,128,115.465397
2,2024-12-03,133,116.374510
3,2024-12-04,172,118.533666
4,2024-12-05,101,116.959238


##### Métricas do Prophet em dezembro

Vamos calcular o MAE e o RMSE entre as vendas reais e as previsões para
dezembro. As métricas são expressas em unidades vendidas por dia.

In [79]:
mae_prophet_dezembro = mean_absolute_error(
    comparacao_prophet_dezembro["y"],
    comparacao_prophet_dezembro["yhat"]
)

In [80]:
rmse_prophet_dezembro = mean_squared_error(
    comparacao_prophet_dezembro["y"],
    comparacao_prophet_dezembro["yhat"]
) ** 0.5

In [81]:
print("MAE:", mae_prophet_dezembro)
print("RMSE:", rmse_prophet_dezembro)

MAE: 22.068310216753954
RMSE: 27.273317655422115


#### 5.4 Resumo das métricas do Prophet

A tabela reúne o MAE e o RMSE de cada mês. As métricas estão em unidades
vendidas por dia e serão usadas depois para comparar com o XGBoost.

In [82]:
resumo_prophet = pd.DataFrame({
    "mes": ["Outubro", "Novembro", "Dezembro"],
    "MAE": [
        mae_prophet_outubro,
        mae_prophet_novembro,
        mae_prophet_dezembro
    ],
    "RMSE": [
        rmse_prophet_outubro,
        rmse_prophet_novembro,
        rmse_prophet_dezembro
    ]
})

resumo_prophet

,mes,MAE,RMSE
0,Outubro,23.603608,29.993773
1,Novembro,25.893052,32.445005
2,Dezembro,22.068310,27.273318


In [83]:
# A tabela abaixo apresenta as métricas arredondadas para duas casas
# decimais, para facilitar a leitura e a comparação entre os meses.
resumo_prophet.round(2)

,mes,MAE,RMSE
0,Outubro,23.60,29.99
1,Novembro,25.89,32.45
2,Dezembro,22.07,27.27


## 6. XGBoost: variáveis de treino para outubro

Usaremos o histórico de vendas com defasagens de 1 e 7 dias, além do dia
da semana e do mês. Primeiro, vamos conferir as colunas disponíveis no
treino de outubro.

In [84]:
treino_outubro.columns

Index(['units_sold', 'promotion_flag'], dtype='str')

### Preparação da série de treino de outubro

A data está no índice de `treino_outubro`. Vamos colocá-la em uma coluna
para facilitar a criação das variáveis de calendário e das defasagens.

In [85]:
dados_xgb_treino_outubro = treino_outubro.reset_index()

In [86]:
dados_xgb_treino_outubro.columns

Index(['date', 'units_sold', 'promotion_flag'], dtype='str')

### Defasagens de vendas no treino de outubro

Vamos criar `lag_1d` e `lag_7d` a partir de `units_sold`. As primeiras linhas
ficarão sem valores para essas variáveis, pois ainda não haverá histórico
suficiente.

In [87]:
dados_xgb_treino_outubro["lag_1d"] = (
    dados_xgb_treino_outubro["units_sold"].shift(1)
)

dados_xgb_treino_outubro["lag_7d"] = (
    dados_xgb_treino_outubro["units_sold"].shift(7)
)

### Variáveis de calendário do treino de outubro

Vamos extrair o dia da semana e o mês de cada data. Essas informações
também estarão disponíveis para as datas que serão previstas.

In [88]:
dados_xgb_treino_outubro["dia_semana"] = (
    dados_xgb_treino_outubro["date"].dt.dayofweek
)

dados_xgb_treino_outubro["mes"] = (
    dados_xgb_treino_outubro["date"].dt.month
)

C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\1180305305.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dados_xgb_treino_outubro["date"].dt.dayofweek


### Remoção de linhas sem histórico suficiente

As primeiras linhas não têm valores para os dois lags. Vamos removê-las do
treino do XGBoost, pois o modelo precisa de todos os atributos preenchidos.

In [89]:
dados_xgb_treino_outubro = dados_xgb_treino_outubro.dropna().copy()

##### Conferência dos atributos de treino

Vamos verificar o tamanho do conjunto preparado e confirmar que os lags e as
variáveis de calendário estão preenchidos.

In [90]:
dados_xgb_treino_outubro[
    ["units_sold", "lag_1d", "lag_7d", "dia_semana", "mes"]
].isna().sum()

units_sold    0
lag_1d        0
lag_7d        0
dia_semana    0
mes           0
dtype: int64

#### Outubro:

##### Separação dos atributos e do alvo

O modelo usará os dois lags e as variáveis de calendário como entrada.
A variável que queremos prever é `units_sold`.

In [91]:
X_treino_outubro = dados_xgb_treino_outubro[
    ["lag_1d", "lag_7d", "dia_semana", "mes"]
]

In [92]:
y_treino_outubro = dados_xgb_treino_outubro["units_sold"]

##### Configuração do XGBoost para outubro

Vamos criar um regressor para prever `units_sold`. Nesta primeira versão,
usaremos as configurações padrão do modelo, definindo explicitamente a
função de perda para regressão.

In [93]:
modelo_xgb_outubro = XGBRegressor(
    objective="reg:squarederror"
)

##### Treinamento do XGBoost para outubro

O modelo será ajustado com os lags e as variáveis de calendário preparados
para o período de treino.

In [94]:
modelo_xgb_outubro.fit(
    X_treino_outubro,
    y_treino_outubro
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


##### Histórico inicial para a previsão recursiva de outubro

A cada dia previsto, usaremos a previsão gerada para atualizar o histórico.
Assim, os lags dos dias seguintes não dependerão das vendas reais de outubro.

In [95]:
historico_xgb_outubro = treino_outubro["units_sold"].tolist()

##### Previsão recursiva do XGBoost para outubro

As previsões serão feitas em sequência. Para cada data, o modelo usará
`lag_1d`, `lag_7d`, `dia_semana` e `mes`. A previsão de cada dia será
adicionada ao histórico usado nos dias seguintes.

In [96]:
previsoes_xgb_outubro = []

for data in teste_outubro.index:
    atributos_dia = pd.DataFrame({
        "lag_1d": [historico_xgb_outubro[-1]],
        "lag_7d": [historico_xgb_outubro[-7]],
        "dia_semana": [data.dayofweek],
        "mes": [data.month]
    })

    previsao_dia = modelo_xgb_outubro.predict(atributos_dia)[0]
    previsoes_xgb_outubro.append(previsao_dia)
    historico_xgb_outubro.append(previsao_dia)

C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\1813560718.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\1813560718.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\1813560718.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\1813560718.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_121

##### Comparação entre vendas reais e previstas pelo XGBoost

Vamos reunir as vendas observadas em outubro e as previsões recursivas do
modelo para avaliar seu desempenho.

In [97]:
comparacao_xgb_outubro = pd.DataFrame({
    "ds": teste_outubro.index,
    "y": teste_outubro["units_sold"].values,
    "yhat": previsoes_xgb_outubro
})

##### Métricas do XGBoost em outubro

Vamos calcular MAE e RMSE entre as vendas reais e as previsões recursivas.
As métricas são expressas em unidades vendidas por dia.

In [98]:
mae_xgb_outubro = mean_absolute_error(
    comparacao_xgb_outubro["y"],
    comparacao_xgb_outubro["yhat"]
)

In [99]:
rmse_xgb_outubro = mean_squared_error(
    comparacao_xgb_outubro["y"],
    comparacao_xgb_outubro["yhat"]
) ** 0.5

In [100]:
print("MAE:", mae_xgb_outubro)
print("RMSE:", rmse_xgb_outubro)

MAE: 27.872629165649414
RMSE: 34.45268565479511


#### Novembro:

##### XGBoost: preparação do treino de novembro

Para prever novembro, usaremos os dados observados até 31/10/2024.
Vamos colocar a data em uma coluna para criar os lags e as variáveis
de calendário.

In [101]:
dados_xgb_treino_novembro = treino_novembro.reset_index()

##### Defasagens de vendas no treino de novembro

Vamos criar `lag_1d` e `lag_7d` usando as vendas anteriores.
As linhas iniciais sem histórico suficiente serão removidas depois.

In [102]:
dados_xgb_treino_novembro["lag_1d"] = (
    dados_xgb_treino_novembro["units_sold"].shift(1)
)

dados_xgb_treino_novembro["lag_7d"] = (
    dados_xgb_treino_novembro["units_sold"].shift(7)
)

##### Variáveis de calendário do treino de novembro

Vamos extrair o dia da semana e o mês de cada data. Essas informações
estarão disponíveis também para as datas previstas.

In [103]:
dados_xgb_treino_novembro["dia_semana"] = (
    dados_xgb_treino_novembro["date"].dt.dayofweek
)

dados_xgb_treino_novembro["mes"] = (
    dados_xgb_treino_novembro["date"].dt.month
)

C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\2322769682.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dados_xgb_treino_novembro["date"].dt.dayofweek


##### Remoção de linhas sem histórico suficiente

Vamos remover as linhas iniciais com valores ausentes nos lags, mantendo
somente os registros com todos os atributos disponíveis.

In [104]:
dados_xgb_treino_novembro = dados_xgb_treino_novembro.dropna().copy()

##### Separação dos atributos e do alvo de novembro

O modelo usará os lags e as variáveis de calendário como atributos de entrada.
A variável prevista será `units_sold`.

In [105]:
X_treino_novembro = dados_xgb_treino_novembro[
    ["lag_1d", "lag_7d", "dia_semana", "mes"]
]

In [106]:
y_treino_novembro = dados_xgb_treino_novembro["units_sold"]

##### Treinamento do XGBoost para novembro

O modelo será treinado com os dados disponíveis até 31/10/2024, usando
`lag_1d`, `lag_7d`, `dia_semana` e `mes`.

In [107]:
modelo_xgb_novembro = XGBRegressor(
    objective="reg:squarederror"
)

modelo_xgb_novembro.fit(
    X_treino_novembro,
    y_treino_novembro
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


##### Histórico inicial para a previsão de novembro

Vamos iniciar o histórico com as vendas observadas até 31/10/2024. As
previsões de novembro serão acrescentadas ao histórico conforme forem
geradas.

In [108]:
historico_xgb_novembro = treino_novembro["units_sold"].tolist()

##### Previsão recursiva do XGBoost para novembro

Para cada data, o modelo usará as vendas anteriores do histórico, além do dia
da semana e do mês. A previsão de cada dia será adicionada ao histórico para
calcular os lags dos próximos dias.

In [109]:
previsoes_xgb_novembro = []

for data in teste_novembro.index:
    atributos_dia = pd.DataFrame({
        "lag_1d": [historico_xgb_novembro[-1]],
        "lag_7d": [historico_xgb_novembro[-7]],
        "dia_semana": [data.dayofweek],
        "mes": [data.month]
    })

    previsao_dia = modelo_xgb_novembro.predict(atributos_dia)[0]
    previsoes_xgb_novembro.append(previsao_dia)
    historico_xgb_novembro.append(previsao_dia)

C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\2063516612.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\2063516612.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\2063516612.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\2063516612.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_121

##### Comparação entre vendas reais e previstas pelo XGBoost

Vamos reunir a data, as vendas observadas e as previsões recursivas de
novembro para calcular as métricas.

In [110]:
comparacao_xgb_novembro = pd.DataFrame({
    "ds": teste_novembro.index,
    "y": teste_novembro["units_sold"].values,
    "yhat": previsoes_xgb_novembro
})

##### Métricas do XGBoost em novembro

Vamos calcular o MAE e o RMSE entre as vendas reais e as previsões recursivas
de novembro. As métricas são expressas em unidades vendidas por dia.

In [111]:
mae_xgb_novembro = mean_absolute_error(
    comparacao_xgb_novembro["y"],
    comparacao_xgb_novembro["yhat"]
)

In [112]:
rmse_xgb_novembro = mean_squared_error(
    comparacao_xgb_novembro["y"],
    comparacao_xgb_novembro["yhat"]
) ** 0.5

In [113]:
print("MAE:", mae_xgb_novembro)
print("RMSE:", rmse_xgb_novembro)

MAE: 26.784521102905273
RMSE: 33.67843432341478


#### Dezembro:

#### XGBoost: preparação do treino de dezembro

Para prever dezembro, usaremos os dados observados até 30/11/2024.
Vamos colocar a data em uma coluna para criar os lags e as variáveis
de calendário.

In [114]:
dados_xgb_treino_dezembro = treino_dezembro.reset_index()

##### Defasagens de vendas no treino de dezembro

Vamos criar `lag_1d` e `lag_7d` com as vendas anteriores. As primeiras linhas,
sem histórico suficiente para calcular os lags, serão removidas depois.

In [115]:
dados_xgb_treino_dezembro["lag_1d"] = (
    dados_xgb_treino_dezembro["units_sold"].shift(1)
)

dados_xgb_treino_dezembro["lag_7d"] = (
    dados_xgb_treino_dezembro["units_sold"].shift(7)
)

##### Variáveis de calendário do treino de dezembro

Vamos extrair o dia da semana e o mês de cada data. Essas informações
também estarão disponíveis para as datas previstas.

In [116]:
dados_xgb_treino_dezembro["dia_semana"] = (
    dados_xgb_treino_dezembro["date"].dt.dayofweek
)

dados_xgb_treino_dezembro["mes"] = (
    dados_xgb_treino_dezembro["date"].dt.month
)

C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\2618238751.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dados_xgb_treino_dezembro["date"].dt.dayofweek


##### Remoção de linhas sem histórico suficiente

As primeiras linhas não têm todos os lags calculados. Vamos removê-las para
treinar o modelo somente com registros completos.

In [117]:
dados_xgb_treino_dezembro = dados_xgb_treino_dezembro.dropna().copy()

##### Separação dos atributos e do alvo de dezembro

O modelo usará `lag_1d`, `lag_7d`, `dia_semana` e `mes` como atributos de
entrada. A variável prevista será `units_sold`.

In [118]:
X_treino_dezembro = dados_xgb_treino_dezembro[
    ["lag_1d", "lag_7d", "dia_semana", "mes"]
]

In [119]:
y_treino_dezembro = dados_xgb_treino_dezembro["units_sold"]

##### Treinamento do XGBoost para dezembro

O modelo será treinado com os dados disponíveis até 30/11/2024, usando
`lag_1d`, `lag_7d`, `dia_semana` e `mes`.

In [120]:
modelo_xgb_dezembro = XGBRegressor(
    objective="reg:squarederror"
)

modelo_xgb_dezembro.fit(
    X_treino_dezembro,
    y_treino_dezembro
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


##### Histórico inicial para a previsão recursiva de dezembro

Vamos iniciar o histórico com as vendas observadas até 30/11/2024. As
previsões de dezembro serão acrescentadas conforme forem geradas.

In [121]:
historico_xgb_dezembro = treino_dezembro["units_sold"].tolist()

##### Previsão recursiva do XGBoost para dezembro

As previsões serão feitas em sequência. A previsão de cada dia será
adicionada ao histórico para calcular os lags dos dias seguintes, sem usar
as vendas reais de dezembro.

In [122]:
previsoes_xgb_dezembro = []

for data in teste_dezembro.index:
    atributos_dia = pd.DataFrame({
        "lag_1d": [historico_xgb_dezembro[-1]],
        "lag_7d": [historico_xgb_dezembro[-7]],
        "dia_semana": [data.dayofweek],
        "mes": [data.month]
    })

    previsao_dia = modelo_xgb_dezembro.predict(atributos_dia)[0]
    previsoes_xgb_dezembro.append(previsao_dia)
    historico_xgb_dezembro.append(previsao_dia)

C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\4069379143.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\4069379143.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\4069379143.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_12108\4069379143.py:7: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "dia_semana": [data.dayofweek],
C:\Users\Yukio Kitamura\AppData\Local\Temp\ipykernel_121

##### Comparação entre vendas reais e previstas pelo XGBoost

Vamos reunir as vendas observadas em dezembro e as previsões recursivas do
modelo.

In [123]:
comparacao_xgb_dezembro = pd.DataFrame({
    "ds": teste_dezembro.index,
    "y": teste_dezembro["units_sold"].values,
    "yhat": previsoes_xgb_dezembro
})

##### Métricas do XGBoost em dezembro

Vamos calcular o MAE e o RMSE entre as vendas reais e as previsões recursivas
de dezembro. As métricas são expressas em unidades vendidas por dia.

In [124]:
mae_xgb_dezembro = mean_absolute_error(
    comparacao_xgb_dezembro["y"],
    comparacao_xgb_dezembro["yhat"]
)

In [125]:
rmse_xgb_dezembro = mean_squared_error(
    comparacao_xgb_dezembro["y"],
    comparacao_xgb_dezembro["yhat"]
) ** 0.5

In [126]:
print("MAE:", mae_xgb_dezembro)
print("RMSE:", rmse_xgb_dezembro)

MAE: 31.443510055541992
RMSE: 38.96882518036052


## 7. Comparação entre Prophet e XGBoost

Vamos reunir as métricas dos dois modelos em uma única tabela, mês a mês,
para identificar qual apresentou menores erros no período de teste.

In [127]:
resumo_xgb = pd.DataFrame({
    "mes": ["Outubro", "Novembro", "Dezembro"],
    "MAE": [
        mae_xgb_outubro,
        mae_xgb_novembro,
        mae_xgb_dezembro
    ],
    "RMSE": [
        rmse_xgb_outubro,
        rmse_xgb_novembro,
        rmse_xgb_dezembro
    ]
})

resumo_xgb

,mes,MAE,RMSE
0,Outubro,27.872629,34.452686
1,Novembro,26.784521,33.678434
2,Dezembro,31.443510,38.968825


In [128]:
# MÉTRICAS ARREDONDADAS PARA FACILITAR A LEITURA E A COMPARAÇÃO
comparacao_modelos = pd.DataFrame({
    "mes": ["Outubro", "Novembro", "Dezembro"],
    "MAE_Prophet": resumo_prophet["MAE"].round(2),
    "RMSE_Prophet": resumo_prophet["RMSE"].round(2),
    "MAE_XGBoost": resumo_xgb["MAE"].round(2),
    "RMSE_XGBoost": resumo_xgb["RMSE"].round(2)
})

comparacao_modelos

,mes,MAE_Prophet,RMSE_Prophet,MAE_XGBoost,RMSE_XGBoost
0,Outubro,23.60,29.99,27.87,34.45
1,Novembro,25.89,32.45,26.78,33.68
2,Dezembro,22.07,27.27,31.44,38.97


## 8. Interpretação dos resultados

O Prophet apresentou os menores valores de MAE e RMSE nos três meses avaliados. Neste experimento, foi o modelo com melhor desempenho para a previsão diária de vendas da SKU YO-029.

O XGBoost foi usado como modelo de comparação. Apesar de sua flexibilidade, não superou o Prophet neste protocolo de avaliação. Essa conclusão se limita à SKU, ao período de teste e às configurações utilizadas; não permite afirmar que o Prophet será superior em outros produtos ou períodos.

### Limitações

- A avaliação abrange uma única SKU e três meses de teste.
- As promoções não foram incluídas como variáveis explicativas.
- A previsão recursiva do XGBoost utiliza previsões anteriores para gerar os dias seguintes, o que pode propagar erros ao longo do mês.
- Os resultados representam erros de previsão diária e, por si só, não determinam um estoque de segurança. Para isso, seriam necessários dados e premissas sobre o prazo de reposição e a demanda durante esse prazo.